# Phase 8 — Bounded recovery policy engine

XGBoost supplies calibrated candidate probabilities. Recovery Policy V3 remains the single decision authority: it applies failure diagnosis, eligibility, support, stopping rules, cooldowns, intervention budgets, and risk-adjusted expected value before producing a dry-run action.

This notebook audits the frozen synthetic simulation. It does not authorize Razorpay execution.

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import pandas as pd
import yaml


def find_repo_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "ml" / "reports" / "phase8" / "policy_engine_metrics.json").exists():
            return candidate
    raise FileNotFoundError("Run `python -m ml.src.policy_engine_simulator` first")


ROOT = find_repo_root()
REPORTS = ROOT / "ml" / "reports" / "phase8"
metrics = json.loads((REPORTS / "policy_engine_metrics.json").read_text())
manifest = json.loads((REPORTS / "phase8_report_manifest.json").read_text())
decisions = pd.read_csv(REPORTS / "policy_engine_decisions.csv")
cases = pd.read_csv(REPORTS / "policy_engine_cases.csv")
policy = yaml.safe_load((ROOT / "ml" / "config" / "intervention_policy.yaml").read_text())
matrix = yaml.safe_load((ROOT / "ml" / "config" / "action_matrix.yaml").read_text())

print("Cases:", len(cases))
print("Decisions:", len(decisions))
print("Execution mode:", metrics["execution_mode"])
print("Razorpay API calls:", metrics["razorpay_api_calls"])

In [ ]:
baseline_rows = []
for name, values in metrics["baselines"].items():
    baseline_rows.append({"policy": name, **values})
baselines = pd.DataFrame(baseline_rows).set_index("policy")
v3 = pd.Series(metrics["recovery_policy_v3"], name="recovery_policy_v3")
display(baselines[["recovery_rate", "recovered_amount", "intervention_count", "intervention_cost", "recovery_roi"]])
display(v3[["recovery_rate", "recovered_amount", "intervention_count", "intervention_cost", "recovery_roi"]])

In [ ]:
comparison = baselines["recovery_rate"].copy()
comparison.loc["recovery_policy_v3"] = metrics["recovery_policy_v3"]["recovery_rate"]
ax = comparison.sort_values().plot.barh(
    figsize=(8, 5),
    title="Synthetic recovery rate by policy",
)
ax.set(xlabel="Recovery rate", ylabel="Policy", xlim=(0, 1))
plt.tight_layout()
plt.show()

In [ ]:
safety = pd.Series({
    "policy_violations": v3["policy_violations"],
    "fraud_cases": v3["fraud_cases"],
    "fraud_automated_actions": v3["fraud_automated_actions"],
    "blocked_decisions": v3["blocked_decisions"],
    "stop_decisions": v3["stop_decisions"],
    "stop_rate": v3["stop_rate"],
    "fallback_decisions": v3["fallback_decisions"],
})
safety

assert v3["policy_violations"] == 0
assert v3["fraud_automated_actions"] == 0
assert metrics["razorpay_api_calls"] == 0
assert metrics["qwen_used"] is False

In [ ]:
print("Configured intervention controls")
controls = []
for action, values in policy["interventions"].items():
    controls.append({"action": action, **values})
pd.DataFrame(controls).set_index("action")

In [ ]:
eligibility_matrix = pd.DataFrame(matrix["failure_classes"]).T
eligibility_matrix

In [ ]:
print("Selected actions across all bounded steps")
display(decisions["selected_action"].fillna("no_action").value_counts())
print("Decision types")
display(decisions["decision_type"].value_counts())
print("Failure classes")
display(decisions["failure_class"].value_counts())

In [ ]:
efficiency = pd.Series({
    "revenue_at_risk": v3["revenue_at_risk"],
    "recovered_amount": v3["recovered_amount"],
    "intervention_count": v3["intervention_count"],
    "actions_per_case": v3["actions_per_case"],
    "intervention_cost": v3["intervention_cost"],
    "recovered_amount_per_intervention": v3["recovered_amount_per_intervention"],
    "recovery_roi": v3["recovery_roi"],
})
efficiency

In [ ]:
sample_columns = [
    "payment_id",
    "decision_number",
    "decision_type",
    "selected_action",
    "failure_class",
    "expected_values",
    "fallback_used",
    "risk_checks_passed",
    "reasons",
]
decisions[sample_columns].sample(8, random_state=42)

In [ ]:
multi_step = cases.loc[cases["intervention_count"].gt(1)]
print("Multi-step cases:", len(multi_step))
if len(multi_step):
    display(multi_step[["payment_id", "recovered", "intervention_count", "terminal_state", "state_history"]].head(10))

## Interpretation

- Recovery Policy V3 is an auditable business policy, not a new model. XGBoost V1 remains frozen.
- V3 uses calibrated probabilities only after deterministic failure diagnosis, eligibility, support, attempt limits, cooldowns, and stopping rules.
- Selection maximizes risk-adjusted expected value rather than probability alone.
- Intervention costs and contact/opt-out availability are explicit synthetic assumptions.
- The simulator limits each action to one potential outcome because Phase 4 contains only one counterfactual draw per payment/action.
- The backend endpoint persists only signed-off dry-run decisions and `would_execute` interventions. It does not call Razorpay.
- Online temporal feature parity is not yet available from the production database, so live model inference remains fail-closed.